In [1]:
import json
import pandas as pd

with open("raw_data/abcd_v1.1.json") as f:
    data = json.load(f)

print({split: len(chats) for split, chats in data.items()})

{'train': 8034, 'dev': 1004, 'test': 1004}


In [2]:
rows = []
for split in ["train", "dev", "test"]:
    for chat in data[split]:
        turns = chat["original"]
        rows.append({
            "split": split,
            "id": chat["convo_id"],
            "flow": chat["scenario"].get("flow"),
            "subflow": chat["scenario"].get("subflow"),
            "first_message": next((t[1] for t in turns if t[0] == "customer"), ""),
            "actions": [t[1] for t in turns if t[0] == "action"],
        })
abcd = pd.DataFrame(rows)
abcd["refund"] = abcd["actions"].apply(lambda acts: any("refund has been made" in a.lower() for a in acts))
abcd.shape

(10042, 7)

In [3]:
abcd.head(20)

,split,id,flow,subflow,first_message,actions,refund
0,train,3592,product_defect,return_size,"Hi! I need to return an item, can you help me ...","[Account has been pulled up for Crystal Minh.,...",False
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund,[Account has been pulled up for Alessandro Pho...,False
2,train,3695,storewide_query,timing_4,HEY HO!,"[Searching the FAQ pages ..., System Action: s...",False
3,train,5798,shipping_issue,manage,"Hello, I would like to change my shipping deat...","[Account has been pulled up for Crystal Minh.,...",False
4,train,3647,storewide_query,pricing_3,Hello. I have a really cool party coming up. A...,"[Searching the FAQ pages ..., System Action: s...",False
5,train,1426,subscription_inquiry,manage_extension,I want an extension for my shopping service.,"[Account has been pulled up for Chloe Zhang., ...",False
6,train,8893,account_access,recover_username,"Hello, I am trying to check the status of my o...",[Account has been pulled up for Alessandro Pho...,False
7,train,5185,troubleshoot_site,shopping_cart,Hello,"[Searching the FAQ pages ..., System Action: s...",False
8,train,10015,shipping_issue,manage,I accidently ordered the wrong item and was wo...,[Account has been pulled up for Norman Bouchar...,True
9,train,1884,shipping_issue,status,I would like to confirm my address is correct ...,"[Account has been pulled up for Crystal Minh.,...",False


In [6]:
def show_chat(chat_id):
    chat = next(c for split in data.values() for c in split if c["convo_id"] == chat_id)
    print(chat["scenario"]["flow"], "/", chat["scenario"]["subflow"])
    for speaker, text in chat["original"]:
        print(f"{speaker[:4].upper():5}{text}")

show_chat(abcd.id.iloc[8])

shipping_issue / manage
AGEN Thank you for contacting Acme Brands! How can I help you today?
CUST I accidently ordered the wrong item and was wondering if I can change the item
AGEN I can check to see if we can make the adjust to your order. Can I have your name or account ID?
CUST Norman Bouchard
ACTI Account has been pulled up for Norman Bouchard.
CUST Account ID: 4VEJCYVP8U
AGEN Thank you. You need to both remove an item and add a different item to your order?
CUST I think at this point I'd rather just cancel the order
AGEN Okay. Which is your order ID number?
CUST Order ID: 5545781356
ACTI Identity verification in progress ...
AGEN What is the shipping status of your order?
CUST It says in transit
ACTI Shipping status of in transit has been noted.
AGEN What is your membership level with us?
CUST Gold
ACTI Membership level of Gold has been noted.
AGEN Since the order has already shipped, I'm unable to cancel to order. However, I'll be able to refund you right away. Once the shipment